In [1]:
import numpy as np
from iodata import load_one
from gbasis.wrappers import from_iodata
from gbasis.integrals.overlap import overlap_integral
from gbasis.evals.density import *
from grid import GaussChebyshev, BeckeRTransform, MolGrid, BeckeWeights

In [5]:
mol = load_one("../examples/water.fchk")

In [6]:
# 1-electron reduced density matrix (1-RDM) from SCF calculation
rdm = mol.one_rdms.get("scf")


In [7]:
coeffs = mol.mo.coeffs
occs = mol.mo.occs
rdm_raw = np.dot(coeffs * occs, coeffs.T)

In [8]:
mo = mol.mo
# Spin-resolved density matrices
# Alpha
rdm_a = (mo.coeffsa * mo.occsa) @ mo.coeffsa.conj().T

# Beta
rdm_b = (mo.coeffsb * mo.occsb) @ mo.coeffsb.conj().T

# Total density
rdm_total = rdm_a + rdm_b

# Verify consistency
rdm_raw = (mo.coeffs * mo.occs) @ mo.coeffs.conj().T

assert np.allclose(rdm_total, rdm_raw)
assert np.allclose(rdm_total, rdm)

In [9]:
np.allclose(rdm_raw, rdm)

True

In [10]:
# basis function from mol object
basis = from_iodata(mol)

In [11]:
# basis function from mol object
basis = from_iodata(mol)

# AO overlap matrix
S = overlap_integral(basis)

# Number of electrons considering non-orthogonal basis
n_alpha = np.trace(rdm_a @ S)
n_beta = np.trace(rdm_b @ S)

print("Alpha electrons:", n_alpha)
print("Beta electrons:", n_beta)

# Check AO orthonormality
print(np.allclose(mo.coeffsa.T @ S @ mo.coeffsa,
                  np.eye(mo.coeffsa.shape[1])))

Alpha electrons: 4.999999998010127
Beta electrons: 4.999999998010127
False


In [12]:
C = mo.coeffsa
occupied = mo.occsa > 1e-8

C_occ = C[:, occupied]
C_vir = C[:, ~occupied]

S_oo = C_occ.T @ S @ C_occ
S_vv = C_vir.T @ S @ C_vir
S_ov = C_occ.T @ S @ C_vir

print("Occupied-occupied:",
      np.max(np.abs(S_oo - np.eye(S_oo.shape[0]))))

print("Virtual-virtual:",
      np.max(np.abs(S_vv - np.eye(S_vv.shape[0]))))

print("Occupied-virtual:",
      np.max(np.abs(S_ov)))


Occupied-occupied: 9.384193422334874e-10
Virtual-virtual: 2.0017730340860568e-08
Occupied-virtual: 1.999734080111672e-08


In [24]:
# Construct molecular grid
oned = GaussChebyshev(npoints=30)
rgrid = BeckeRTransform(0.0, R=1.5).transform_1d_grid(oned)

grid = MolGrid.from_preset(
    atnums=mol.atnums,
    atcoords=mol.atcoords,
    rgrid=rgrid,
    preset="coarse",
    aim_weights=BeckeWeights(),
    store=True,
)

# Evaluate spin-resolved densities
dens_a = evaluate_density(rdm_a, basis, grid.points)
dens_b = evaluate_density(rdm_b, basis, grid.points)

# Total electron density
moldens_raw = dens_a + dens_b
moldens = evaluate_density(rdm, basis, grid.points)

# Spin density
spin_dens = dens_a - dens_b

In [25]:
print("Alpha electrons:", grid.integrate(dens_a))
print("Beta electrons:", grid.integrate(dens_b))
print("Total electrons:", grid.integrate(moldens))
print("Total electrons (raw):", grid.integrate(moldens_raw))
print("Spin polarization:", grid.integrate(spin_dens))

Alpha electrons: 5.000012844028549
Beta electrons: 5.000012844028549
Total electrons: 10.000025689713965
Total electrons (raw): 10.000025688057098
Spin polarization: 0.0


In [26]:
print("MO type:", mo.kind)
print("Maximum spin density:", np.max(spin_dens))
print("Minimum spin density:", np.min(spin_dens))
print("Integrated absolute spin density:",
      grid.integrate(np.abs(spin_dens)))

MO type: unrestricted
Maximum spin density: 0.0
Minimum spin density: 0.0
Integrated absolute spin density: 0.0


In [27]:
spin_dens_true = mol.one_rdms.get("scf_spin")
spin_dens_true = evaluate_density(spin_dens_true, basis, grid.points)
print("Integrated absolute spin density:",
      grid.integrate(np.abs(spin_dens_true)))

Integrated absolute spin density: 0.0


### Test Difference between Restricted and Unrestricted Functionals

In [ ]:
res_mol = load_one("../examples/0001_q000_m01_k00_force_b3lyp_def2tzvpd.fchk")
unr_mol = load_one("../examples/0001_q000_m01_k00_force_ub3lyp_def2tzvpd.fchk")

res_basis = from_iodata(res_mol)
unr_basis = from_iodata(unr_mol)

In [ ]:
res_mo = res_mol.mo
unr_mo = unr_mol.mo

In [ ]:
res_rdm_a = (res_mol.mo.coeffsa * res_mol.mo.occsa) @ res_mol.mo.coeffsa.conj().T
res_rdm_b = (res_mol.mo.coeffsb * res_mol.mo.occsb) @ res_mol.mo.coeffsb.conj().T

unr_rdm_a = (unr_mol.mo.coeffsa * unr_mol.mo.occsa) @ unr_mol.mo.coeffsa.conj().T
unr_rdm_b = (unr_mol.mo.coeffsb * unr_mol.mo.occsb) @ unr_mol.mo.coeffsb.conj().T

In [ ]:
oned = GaussChebyshev(npoints=100)
rgrid = BeckeRTransform(0.0, R=1.5).transform_1d_grid(oned)

res_grid = MolGrid.from_preset(
    atnums=res_mol.atnums,
    atcoords=res_mol.atcoords,
    rgrid=rgrid,
    preset="fine",
    aim_weights=BeckeWeights(),
    store=True,
)

unr_grid = MolGrid.from_preset(
    atnums=unr_mol.atnums,
    atcoords=unr_mol.atcoords,
    rgrid=rgrid,
    preset="fine",
    aim_weights=BeckeWeights(),
    store=True,
)

res_dens_a = evaluate_density(res_rdm_a, res_basis, res_grid.points)
res_dens_b = evaluate_density(res_rdm_b, res_basis, res_grid.points)

unr_dens_a = evaluate_density(unr_rdm_a, unr_basis, unr_grid.points)
unr_dens_b = evaluate_density(unr_rdm_b, unr_basis, unr_grid.points)

/home/merreeves/ffprime/.ffprime/lib/python3.12/site-packages/grid/atomgrid.py:905: UserWarning: Lebedev weights are negative which can introduce round-off errors.
  sphere_grid = AngularGrid(degree=deg_i, method=method)
/home/merreeves/ffprime/.ffprime/lib/python3.12/site-packages/grid/atomgrid.py:905: UserWarning: Lebedev weights are negative which can introduce round-off errors.
  sphere_grid = AngularGrid(degree=deg_i, method=method)


In [ ]:
print(np.allclose(res_dens_a, unr_dens_a))
print(np.allclose(res_dens_b, unr_dens_b))

True
True


### Taylor Expansion Approximation of Exchange/Fermi Hole:

Apply the variable transformation $\textbf{r} = \textbf{r}_1$ and $\textbf{s} = \textbf{r}_2 - \textbf{r}_1$. The approxmiation of the Fermi hole is a spherical average over $\textbf{s}$ about the reference point $\textbf{r}$

$$
h_{X\sigma}(\textbf{r}, \textbf{r+s}) = - \frac{|\rho_\sigma(\textbf{r}, \textbf{r+s})|^2}{\rho_\sigma(\textbf{r})} \approx - \rho_\sigma(\textbf{r}) - Q_\sigma(\textbf{r}) s^2.
$$

where 

$$
Q_\sigma(\textbf{r}) = \frac{1}{6} \left[ \nabla^2 \rho_\sigma(\textbf{r}) - 2\tau_\sigma + \frac{1}{2} \frac{(\nabla\rho_\sigma(\textbf{r}))^2}{\rho_\sigma(\textbf{r})} \right]
$$

Because the exchange energy only depends on the spherical average of the Fermi hole, the approximation of the Fermi hole is restricted to spherically symmetric models.

The probability of finding another same-spin electron at the same position as the reference electron is completely extingushed
$$
h_{X\sigma}(\textbf{r}, \textbf{r}) = - \rho_\sigma(\textbf{r})
$$

the exchange hole at any reference point in any inhomogenenous system can be calculated by taking the spherically averaged exchange hole of the 1s electron through a parameterization.

...
...
...

$a$ and $b$ can be solved for by finding $x=ab$ that satisfies the nonlinear one-dimensional equation 

$$
\frac{xe^{-2x/3}}{x-2} = \frac{2}{3} \pi^{2/3} \frac{\rho_\sigma^{5/3}}{Q_\sigma}
$$

$b$ is then given by

$$
b^3 = \frac{x^3e^{-x}}{8 \pi \rho_\sigma}
$$

and $a = x/b$

In [3]:
mol = load_one("../examples/water.fchk")
rdm = mol.one_rdms.get("scf")
basis = from_iodata(mol)
mo = mol.mo
rdm_alpha = (mo.coeffsa * mo.occsa) @ mo.coeffsa.conj().T
rdm_beta = (mo.coeffsb * mo.occsb) @ mo.coeffsb.conj().T

oned = GaussChebyshev(npoints=30)
rgrid = BeckeRTransform(0.0, R=1.5).transform_1d_grid(oned)

grid = MolGrid.from_size(
    atnums=mol.atnums,
    atcoords=mol.atcoords,
    rgrid=rgrid,
    size=38,
    aim_weights=BeckeWeights(),
    store=True,
)

dens_alpha = evaluate_density(rdm_alpha, basis, grid.points)
grad_alpha = evaluate_density_gradient(rdm_alpha, basis, grid.points)
lap_alpha = evaluate_density_laplacian(rdm_alpha, basis, grid.points)
tau_alpha = evaluate_posdef_kinetic_energy_density(rdm_alpha, basis, grid.points)

dens_beta = evaluate_density(rdm_beta, basis, grid.points)
grad_beta = evaluate_density_gradient(rdm_beta, basis, grid.points)
lap_beta = evaluate_density_laplacian(rdm_beta, basis, grid.points)
tau_beta = evaluate_posdef_kinetic_energy_density(rdm_beta, basis, grid.points)

/home/merreeves/ffprime/.ffprime/lib/python3.12/site-packages/grid/molgrid.py:436: RuntimeWarning: Sizes are used for making the angular grids, degrees are ignored!
  AtomGrid(rad_grid, degrees=None, sizes=[size], center=atcoord, rotate=rotate)


In [4]:
from ffprime.nonbonding.vdw import XDMParameters

xdm_test = XDMParameters(None, None, None)
a_alpha, b_alpha = xdm_test.compute_hole_parameters(dens_alpha, grad_alpha, lap_alpha, tau_alpha)
a_beta, b_beta = xdm_test.compute_hole_parameters(dens_beta, grad_beta, lap_beta, tau_beta)

print("Alpha hole parameters shape: a =", a_alpha.shape, ", b =", b_alpha.shape)
print("Beta hole parameters shape: a =", a_beta.shape, ", b =", b_beta.shape)

print("Maximum a_alpha:", np.max(a_alpha))
print("Maximum b_alpha:", np.max(b_alpha))
print("Maximum a_beta:", np.max(a_beta))
print("Maximum b_beta:", np.max(b_beta))

print("Minimum a_alpha:", np.min(a_alpha))
print("Minimum b_alpha:", np.min(b_alpha))
print("Minimum a_beta:", np.min(a_beta))
print("Minimum b_beta:", np.min(b_beta))

Alpha hole parameters shape: a = (3420,) , b = (3420,)
Beta hole parameters shape: a = (3420,) , b = (3420,)
Maximum a_alpha: 30.33528726164372
Maximum b_alpha: 9.548942728281117
Maximum a_beta: 30.33528726164372
Maximum b_beta: 9.548942728281117
Minimum a_alpha: 1.3701914626601406
Minimum b_alpha: 0.06592981905033164
Minimum a_beta: 1.3701914626601406
Minimum b_beta: 0.06592981905033164
